<pre>
IEX_20260925.pcap.gz
        │
        │ gzip decompression on the fly
        ▼
      PCAP
        │
        ▼
     packets
        │
        ▼
     IEX TOPS
        │
        ▼
   T trade messages
        │
        ├── Stock 1 → parquet
        ├── Stock 2 → parquet
        ├── Stock 3 → parquet
        ├── Stock 4 → parquet
        └── Stock 5 → parquet
</pre>

In [1]:
import gzip
import dpkt
import struct
from collections import defaultdict
import pandas as pd
import os
import pyarrow as pa
import pyarrow.parquet as pq
import glob

In [3]:
gz_path = r"C:\IEX\cache\data_feeds_*.pcap.gz"

volume_by_symbol = defaultdict(int)
trades_by_symbol = defaultdict(int)

file_path = glob.glob(gz_path)[0]
MAX_PACKETS = 10_000_000

with gzip.open(file_path, "rb") as f:
    pcap = dpkt.pcapng.Reader(f)

    for packet_no, (pcap_ts, raw_packet) in enumerate(pcap):
        if packet_no >= MAX_PACKETS:
            break
        eth = dpkt.ethernet.Ethernet(raw_packet)
        ip = eth.data
        if not isinstance(ip, dpkt.ip.IP):
            continue
        udp = ip.data
        if not isinstance(udp, dpkt.udp.UDP):
            continue

        # IEX UDP payload begins after:
        # Ethernet 14 + IPv4 20 + UDP 8 = 42
        payload = udp.data
        if len(payload) < 40:
            continue

        message_count = struct.unpack_from("<H", payload, 14)[0]
        data = payload[40:]
        pos = 0

        for _ in range(message_count):
            if pos + 2 > len(data):
                break
            msg_len = struct.unpack_from(
                "<H", data, pos
            )[0]
            msg_start = pos + 2
            msg_end = msg_start + msg_len
            if msg_end > len(data):
                break
            msg = data[msg_start:msg_end]
            if len(msg) >= 38 and msg[0] == ord("T"):
                symbol = msg[10:18].decode(
                    "ascii", errors="ignore"
                ).rstrip("\x00").strip()
                size = struct.unpack_from("<I", msg, 18)[0]
                volume_by_symbol[symbol] += size
                trades_by_symbol[symbol] += 1

            pos = msg_end

In [4]:
stats = pd.DataFrame({
    "symbol": list(volume_by_symbol.keys()),
    "volume": list(volume_by_symbol.values()),
    "trades": [
        trades_by_symbol[s]
        for s in volume_by_symbol.keys()
    ]
})

stats = stats.sort_values("volume", ascending=False).reset_index(drop=True)
print(stats.head(10))

  symbol   volume  trades
0      T  1201304    2732
1     VZ   464366    1977
2   BITO   394388     119
3   SOXS   333606    2589
4   AAPL   308538    3686
5   PLTD   289852      96
6   ALHC   242703    1413
7    NOK   238841     360
8   CRGY   197268    1044
9   SPCX   174032    1415


In [ ]:
print(stats.head(20))

In [ ]:
df1 = pd.read_parquet("C:\IEX\data\20161212\AAPL_quotes_0001.parquet")

In [ ]:
df1